# 📉 Notebook 3 — ARIMA Model (Baseline)
**Goal:** Fit a plain ARIMA model as a statistical baseline.

- Uses `pmdarima.auto_arima` to automatically select best (p, d, q)
- Forecasts the Oct–Dec 2025 test set
- Saves metrics to `models/` for comparison

**Contents:**
1. Setup & Data
2. Auto-select ARIMA order (AIC)
3. Fit & Summary
4. Forecast vs Actual
5. Residual Analysis
6. Metrics


## 1. Setup & Data

In [ ]:
import os, warnings, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
import pmdarima as pm
warnings.filterwarnings("ignore")
os.makedirs("plots", exist_ok=True); os.makedirs("models", exist_ok=True)

DATA_PATH  = "data/UPI_Master_2021_2025.csv"
TRAIN_END  = "2025-09-30"; TEST_START = "2025-10-01"
BLUE,RED,ORANGE,GREEN = "#1A6FBF","#D62728","#E07B39","#2CA02C"
plt.rcParams.update({"figure.facecolor":"#FAFAFA","axes.facecolor":"#FAFAFA",
    "axes.grid":True,"grid.alpha":0.3,"font.family":"DejaVu Sans",
    "axes.spines.top":False,"axes.spines.right":False})

def evaluation_metrics(actual, predicted):
    a, p = np.array(actual), np.array(predicted)
    return {"MAE":round(np.mean(np.abs(a-p)),4),
            "RMSE":round(np.sqrt(np.mean((a-p)**2)),4),
            "MAPE":round(np.mean(np.abs((a-p)/a))*100,4)}

df    = pd.read_csv(DATA_PATH, parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)
train = df[df["Date"]<=TRAIN_END]; test = df[df["Date"]>=TEST_START]
print(f"Train: {len(train):,} rows  |  Test: {len(test):,} rows")


## 2. Auto-select ARIMA order — Volume

In [ ]:
TARGET = "Volume (In Mn.)"   # ← change to "Value (In Cr.)" to run on value

train_series = train.set_index("Date")[TARGET]
test_series  = test.set_index("Date")[TARGET]

print(f"Running auto_arima for: {TARGET} ...")
auto_model = pm.auto_arima(
    train_series,
    start_p=0, start_q=0, max_p=5, max_q=5,
    d=None, seasonal=False,
    information_criterion="aic", stepwise=True,
    suppress_warnings=True, error_action="ignore", trace=True,
)
order = auto_model.order
print(f"\nBest order: ARIMA{order}  |  AIC: {auto_model.aic():.2f}")


## 3. Fit ARIMA & Print Summary

In [ ]:
model  = ARIMA(train_series, order=order)
fitted = model.fit()
print(fitted.summary())


## 4. Forecast vs Actual

In [ ]:
forecast_result = fitted.get_forecast(steps=len(test_series))
forecast_mean   = forecast_result.predicted_mean
conf_int        = forecast_result.conf_int(alpha=0.05)
forecast_mean.index = test_series.index
conf_int.index      = test_series.index

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(train_series.index[-120:], train_series.values[-120:],
        color=BLUE, lw=1.2, label="Train (last 120 days)")
ax.plot(test_series.index, test_series.values, color=GREEN, lw=1.5, label="Actual (Test)")
ax.plot(forecast_mean.index, forecast_mean.values, color=RED, lw=1.5,
        linestyle="--", label=f"ARIMA{order} Forecast")
ax.fill_between(conf_int.index, conf_int.iloc[:,0], conf_int.iloc[:,1],
                color=RED, alpha=0.1, label="95% CI")
ax.set_title(f"ARIMA{order} — {TARGET} Forecast vs Actual", fontsize=13, fontweight="bold")
ax.set_ylabel(TARGET); ax.legend()
plt.tight_layout()
plt.savefig(f"plots/15_arima_{TARGET[:3].lower()}.png", dpi=150, bbox_inches="tight")
plt.show()


## 5. Residual Analysis

In [ ]:
residuals = test_series.values - forecast_mean.values

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(test_series.index, residuals, color=ORANGE, lw=1.0)
axes[0].axhline(0, color="black", lw=1, linestyle="--")
axes[0].fill_between(test_series.index, residuals, 0, alpha=0.2, color=ORANGE)
axes[0].set_title("Residuals over Time", fontweight="bold")

import seaborn as sns
sns.histplot(residuals, bins=30, kde=True, ax=axes[1], color=ORANGE)
axes[1].axvline(0, color="black", lw=1.5, linestyle="--")
axes[1].set_title("Residual Distribution", fontweight="bold")
plt.tight_layout(); plt.show()

print(f"Residual mean : {residuals.mean():.4f}  (close to 0 = unbiased)")
print(f"Residual std  : {residuals.std():.4f}")


## 6. Metrics & Save

In [ ]:
metrics = evaluation_metrics(test_series.values, forecast_mean.values)
print(f"Test Set Metrics — ARIMA{order} on {TARGET}")
print(f"  MAE  : {metrics['MAE']:.4f}")
print(f"  RMSE : {metrics['RMSE']:.4f}")
print(f"  MAPE : {metrics['MAPE']:.2f}%")

with open(f"models/arima_{TARGET[:3].lower()}_metrics.json","w") as f:
    json.dump({"model":f"ARIMA{order}","target":TARGET,**metrics},f,indent=2)
print("\n✅ Metrics saved to models/")
